In [1]:
#生成phoneme表
phonem_path='/root/Documents/MFA/LibriTTS/dictionary/phones/phones.txt'
phonem_dict={}
with open(phonem_path,'r') as f:
    for line in f:
        phon,phon_idx=line.strip().split()
        phonem_dict[phon]=int(phon_idx)
phone_list=[]
for key in phonem_dict.keys():
    phone_list.append(key)
phone_list_save_path='/nas-wulanchabu/jiahao.mei/code/x2audio/data/libritts/phone_set.json'
import json
with open(phone_list_save_path,'w') as f:
    json.dump(phone_list,f)
print(f'len of phone_list:{len(phone_list)}')

len of phone_list:92


In [29]:
# 定义textured读取函数    
import textgrid
import re

def extract_phones_and_durations(file_path):
    tg = textgrid.TextGrid.fromFile(file_path)
    spkid_match = re.search(r'/([^/]+)_\d+_\d+_\d+.TextGrid$', file_path)
    spkid = spkid_match.group(1) if spkid_match else "0"
    
    phones = []
    durations = []
    
    for tier in tg:
        if tier.name == "phones":
            for interval in tier:
                if interval.mark.strip():  # Ignore empty text entries
                    phon=interval.mark.strip()
                    if phon=='':
                        phon='sil'
                    phones.append(interval.mark.strip())
                    durations.append(interval.maxTime - interval.minTime)
    
    return phones, durations, spkid

# 示例输入（读取 TextGrid 文件）
file_path = "/nas-wulanchabu/jiahao.mei/data/tts/LibriTTS/textgrid/dev-clean/84/121123/0084_121123_000008_000003.TextGrid"
phones, durations, spkid= extract_phones_and_durations(file_path)

# 输出结果
assert len(phones) == len(durations)
print("Phonemes:", phones)
print("Durations:", durations)
print("Speaker ID:", spkid)



Phonemes: ['HH', 'UW1', 'AA1', 'R', 'Y', 'UW1', 'S', 'ER1', 'HH', 'IY1', 'AE1', 'S', 'T', 'DH', 'AE1', 'T', 'F', 'ER0', 'G', 'EH1', 'T', 'DH', 'AH0', 'T', 'DH', 'IH1', 'S', 'IH0', 'Z', 'N', 'AA1', 'T', 'DH', 'AH0', 'M', 'AE1', 'N', 'ER0', 'T', 'AH0', 'EH1', 'N', 'T', 'ER0', 'AH0', 'HH', 'AW1', 'S', 'S', 'T', 'R', 'IH1', 'K', 'AH0', 'N', 'W', 'IH0', 'TH', 'D', 'EH1', 'TH', 'G', 'OW1', 'S', 'ER1', 'G', 'OW1', 'B', 'AH1', 'T', 'M', 'AO1', 'R', 'AH0', 'L', 'R', 'IH0', 'M', 'EY1', 'N', 'D', 'M', 'OW1', 'SH', 'AH0', 'N', 'L', 'AH0', 'S', 'HH', 'IY1', 'K', 'UH1', 'D', 'N', 'AA1', 'T', 'D', 'IH0', 'T', 'AE1', 'CH', 'HH', 'IH0', 'Z', 'AY1', 'Z', 'F', 'ER0', 'M', 'DH', 'AE1', 'T', 'D', 'IH0', 'S', 'AO1', 'R', 'D', 'ER0', 'D', 'B', 'EH1', 'D', 'AE1', 'N', 'D', 'DH', 'AH0', 'P', 'EY1', 'L', 'K', 'AO1', 'R', 'P', 'S', 'AH0', 'V', 'DH', 'IY0', 'Y', 'AH1', 'NG', 'G', 'ER1', 'L', 'HH', 'UW1', 'W', 'AH0', 'Z', 'L', 'AY1', 'IH0', 'NG', 'AA1', 'N', 'IH0', 'T']
Durations: [0.09, 0.09, 0.1, 0.06, 0.0899999

In [ ]:
#生成audio.json 和 phon.h5 串行版本
import os
import json
textgrid_dir = "/nas-wulanchabu/jiahao.mei/data/tts/LibriTTS/textgrid"
ouput_base_dir='/nas-wulanchabu/jiahao.mei/code/x2audio/data/libritts'
# train_set_name=['train-clean-100','train-clean-360','train-other-500']
train_set_name=[]
# dev_set_name=['dev-clean','dev-other']
dev_set_name=[]
test_set_name=['test-clean','test-other']

train_audio_jsonl_data=[]
train_phon_jsonl_data=[]
dev_audio_jsonl_data=[]
dev_phon_jsonl_data=[]
test_audio_jsonl_data=[]
test_phone_jsonl_data=[]
phoneme_set_path='/nas-wulanchabu/jiahao.mei/code/x2audio/data/libritts/phone_set.json'
phoneme_set=json.load(open(phoneme_set_path,'r'))
phon2phone_id={phone:idx for idx,phone in enumerate(phoneme_set)} 
split2h5data={
    'train':{
        'id2phone':{},
        'id2duration':{},
        'id2spkid':{},
    },
    'val':{
        'id2phone':{},
        'id2duration':{},
        'id2spkid':{},
    },
    "test":{
        'id2phone':{},
        'id2duration':{},
        'id2spkid':{},
    }
}
cnt=0
for dir,_,file_names in os.walk(textgrid_dir):
    for file in file_names:
        if file.endswith('.TextGrid'):
            print(f'processing {cnt} file:{file}')
            cnt+=1
            #print(f'processing {file}')
            id=file.replace('.TextGrid','')
            textgrid_file=os.path.join(dir,file)
            phonemes, durations,spkid = extract_phonemes_and_durations(textgrid_file)
            #print("Phoneme Sequence:", " ".join(phonemes))
            #print("Duration Sequence:", durations)
            #print("spkid:",spkid)
            #print("file:",file)
            #print("dir:",dir)
            audio_path=textgrid_file.replace('/nas-wulanchabu/jiahao.mei/data/tts/LibriTTS/textgrid','/nas-wulanchabu/jiahao.mei/data/tts/LibriTTS').replace('.TextGrid','.wav')
            if not os.path.exists(audio_path):
                raise ValueError(f'audio file not exists:{audio_path}')
            if dir.split('/')[-3] in train_set_name:
                # save_dir=os.path.join(ouput_base_dir,'train')
                train_audio_jsonl_data.append(
                    {
                        "audio_id": file.replace('.TextGrid',''),
                        "audio": audio_path,
                    }
                )
                train_phon_jsonl_data.append(
                    {
                        "audio_id": file.replace('.TextGrid',''),
                        "audio":"/nas-wulanchabu/jiahao.mei/code/x2audio/data/libritts/train/phone.h5",
                    }
                )
                split2h5data['train']['id2phone'][id]=[phon2phone_id[phone] for phone in phonemes]
                split2h5data['train']['id2duration'][id]=durations
                split2h5data['train']['id2spkid'][id]=spkid
            elif dir.split('/')[-3] in dev_set_name:
                # save_dir=os.path.join(ouput_base_dir,'val')
                dev_audio_jsonl_data.append(
                    {
                        "audio_id": file.replace('.TextGrid',''),
                        "audio": audio_path,
                    }
                )
                dev_phon_jsonl_data.append(
                    {
                        "audio_id": file.replace('.TextGrid',''),
                        "audio":"/nas-wulanchabu/jiahao.mei/code/x2audio/data/libritts/val/phone.h5",
                    }
                )
                split2h5data['val']['id2phone'][id]=[phon2phone_id[phone] for phone in phonemes]
                split2h5data['val']['id2duration'][id]=durations
                split2h5data['val']['id2spkid'][id]=spkid
            elif dir.split('/')[-3] in test_set_name:
                # save_dir=os.path.join(ouput_base_dir,'test')
                test_audio_jsonl_data.append(
                    {
                        "audio_id": file.replace('.TextGrid',''),
                        "audio": audio_path,
                    }
                )
                test_phone_jsonl_data.append(
                    {
                        "audio_id": file.replace('.TextGrid',''),
                        "audio":"/nas-wulanchabu/jiahao.mei/code/x2audio/data/libritts/test/phone.h5",
                    }
                )
                split2h5data['test']['id2phone'][id]=[phon2phone_id[phone] for phone in phonemes]
                split2h5data['test']['id2duration'][id]=durations
                split2h5data['test']['id2spkid'][id]=spkid
            else:
                raise ValueError(f'Invalid dir name:{dir}')
            
def save_jsonl(list_data,save_path):
    with open(save_path,'w') as f:
        for data in list_data:
            f.write(json.dumps(data)+'\n')
train_save_path=os.path.join(ouput_base_dir,'train','audio.jsonl')
save_jsonl(train_audio_jsonl_data,train_save_path)
print(f'save train data to {train_save_path}')
train_phon_save_path=os.path.join(ouput_base_dir,'train','phoneme.jsonl')
save_jsonl(train_phon_jsonl_data,train_phon_save_path)
print(f'save train phoneme data to {train_phon_save_path}')



dev_save_path=os.path.join(ouput_base_dir,'val','audio.jsonl')
json.dump(dev_audio_jsonl_data,open(dev_save_path,'w'))
print(f'save dev data to {dev_save_path}')
dev_phon_save_path=os.path.join(ouput_base_dir,'val','phoneme.jsonl')
json.dump(dev_phon_jsonl_data,open(dev_phon_save_path,'w'))
print(f'save dev phoneme data to {dev_phon_save_path}')

test_save_path=os.path.join(ouput_base_dir,'test','audio.jsonl')
json.dump(test_audio_jsonl_data,open(test_save_path,'w'))
print(f'save test data to {test_save_path}')
test_phon_save_path=os.path.join(ouput_base_dir,'test','phoneme.jsonl')
json.dump(test_phone_jsonl_data,open(test_phon_save_path,'w'))
print(f'save test phoneme data to {test_phon_save_path}')

#保存为h5文件
import h5py 
h5_train_save_path=os.path.join(ouput_base_dir,'train','phone.h5')
with h5py.File(h5_train_save_path,'w') as hf:
    hf.create_group('phoneme')
    hf.create_group('phoneme_duration')
    hf.create_group('spkid')
    for id in split2h5data['train']['id2phone'].keys():
        hf['phoneme'][id]=split2h5data['train']['id2phone'][id]
        hf['phoneme_duration'][id]=split2h5data['train']['id2duration'][id]
        hf['spkid'][id]=split2h5data['train']['id2spkid'][id]
print(f'save train h5 data to {h5_train_save_path}')

h5_val_save_path=os.path.join(ouput_base_dir,'val','phone.h5')
with h5py.File(h5_val_save_path,'w') as hf:
    hf.create_group('phoneme')
    hf.create_group('phoneme_duration')
    hf.create_group('spkid')
    for id in split2h5data['val']['id2phone'].keys():
        hf['phoneme'][id]=split2h5data['val']['id2phone'][id]
        hf['phoneme_duration'][id]=split2h5data['val']['id2duration'][id]
        hf['spkid'][id]=split2h5data['val']['id2spkid'][id] 
print(f'save val h5 data to {h5_val_save_path}')

h5_test_save_path=os.path.join(ouput_base_dir,'test','phone.h5')
with h5py.File(h5_test_save_path,'w') as hf:
    hf.create_group('phoneme')
    hf.create_group('phoneme_duration')
    hf.create_group('spkid')
    for id in split2h5data['test']['id2phone'].keys():
        hf['phoneme'][id]=split2h5data['test']['id2phone'][id]
        hf['phoneme_duration'][id]=split2h5data['test']['id2duration'][id]
        hf['spkid'][id]=split2h5data['test']['id2spkid'][id]
print(f'save test h5 data to {h5_test_save_path}')
            
            

In [ ]:
import os
import json
import concurrent.futures
import h5py
from functools import partial

# 生成audio.json 和h5 文件 并行版本

textgrid_dir = "/nas-wulanchabu/jiahao.mei/data/tts/LibriTTS/textgrid"
ouput_base_dir = '/nas-wulanchabu/jiahao.mei/code/x2audio/data/libritts'

train_set_name=['train-clean-100','train-clean-360','train-other-500']
# train_set_name=[]
dev_set_name=['dev-clean','dev-other']
# dev_set_name=[]
test_set_name=['test-clean','test-other']

phoneme_set_path = '/nas-wulanchabu/jiahao.mei/code/x2audio/data/libritts/phone_set.json'
phoneme_set = json.load(open(phoneme_set_path, 'r'))
phon2phone_id = {phone: idx for idx, phone in enumerate(phoneme_set)}

split2h5data = {
    'train': {'id2phone': {}, 'id2duration': {}, 'id2spkid': {}},
    'val': {'id2phone': {}, 'id2duration': {}, 'id2spkid': {}},
    'test': {'id2phone': {}, 'id2duration': {}, 'id2spkid': {}}
}

def process_textgrid(file, dir):
    if not file.endswith('.TextGrid'):
        return None
    
    id = file.replace('.TextGrid', '')
    textgrid_file = os.path.join(dir, file)
    phonemes, durations, spkid = extract_phonemes_and_durations(textgrid_file)
    audio_path = textgrid_file.replace(textgrid_dir, '/nas-wulanchabu/jiahao.mei/data/tts/LibriTTS').replace('.TextGrid', '.wav')
    
    if not os.path.exists(audio_path):
        raise ValueError(f'Audio file not exists: {audio_path}')
    
    subset = dir.split('/')[-3]
    result = {'audio_id': id, 'audio': audio_path}
    
    phon_id_seq = [phon2phone_id[phone] for phone in phonemes]
    
    if subset in train_set_name:
        phon_result = {'audio_id': id, 'audio': f"{ouput_base_dir}/train/phone.h5"}
        return ('train', result, phon_result, id, phon_id_seq, durations, spkid)
    elif subset in dev_set_name:
        phon_result = {'audio_id': id, 'audio': f"{ouput_base_dir}/val/phone.h5"}
        return ('val', result, phon_result, id, phon_id_seq, durations, spkid)
    elif subset in test_set_name:
        phon_result = {'audio_id': id, 'audio': f"{ouput_base_dir}/test/phone.h5"}
        return ('test', result, phon_result, id, phon_id_seq, durations, spkid)
    else:
        raise ValueError(f'Invalid dir name: {dir}')

def save_h5_data(split, data):
    h5_path = os.path.join(ouput_base_dir, split, 'phone.h5')
    with h5py.File(h5_path, 'w') as hf:
        hf.create_group('phoneme')
        hf.create_group('phoneme_duration')
        hf.create_group('spkid')
        # for id, (phon, dur, spk) in data.items():
        #     hf['phoneme'][id] = phon
        #     hf['phoneme_duration'][id] = dur
        #     hf['spkid'][id] = spk
        for id in data['id2phone'].keys():
            hf['phoneme'][id] = data['id2phone'][id]
            hf['phoneme_duration'][id] = data['id2duration'][id]
            hf['spkid'][id] = data['id2spkid'][id]
    print(f'Saved {split} h5 data to {h5_path}')
def process_textgrid_wrapper(args):
    """包装函数，避免 lambda 造成 PicklingError"""
    return process_textgrid(*args)

# 并行处理 TextGrid 文件
all_files = []  # 这里应该存放 (dir, file) 的元组
for dir, _, file_names in os.walk(textgrid_dir):
    for file in file_names:
        if file.endswith('.TextGrid'):
            all_files.append((file, dir))  # 直接存储参数元组
print(f'Found {len(all_files)} TextGrid files')

train_audio_jsonl_data, train_phon_jsonl_data = [], []
dev_audio_jsonl_data, dev_phon_jsonl_data = [], []
test_audio_jsonl_data, test_phone_jsonl_data = [], []

with concurrent.futures.ProcessPoolExecutor() as executor:
    for result in executor.map(process_textgrid_wrapper, all_files):
        if result:
            split, audio_data, phon_data, id, phon_id_seq, durations, spkid = result
            split2h5data[split]['id2phone'][id] = phon_id_seq
            split2h5data[split]['id2duration'][id] = durations
            split2h5data[split]['id2spkid'][id] = spkid
            # split2h5data[split]['id2spkid'][id] = '0026'
            
            if split == 'train':
                train_audio_jsonl_data.append(audio_data)
                train_phon_jsonl_data.append(phon_data)
            elif split == 'val':
                dev_audio_jsonl_data.append(audio_data)
                dev_phon_jsonl_data.append(phon_data)
            elif split == 'test':
                test_audio_jsonl_data.append(audio_data)
                test_phone_jsonl_data.append(phon_data)

# 保存 JSONL 数据
# def save_jsonl(data, path):
#     with open(path, 'w') as f:
#         json.dump(data, f)
#     print(f'Saved data to {path}')
    
def save_jsonl(list_data,save_path):
    with open(save_path,'w') as f:
        for data in list_data:
            f.write(json.dumps(data)+'\n')

save_jsonl(train_audio_jsonl_data, os.path.join(ouput_base_dir, 'train', 'audio.jsonl'))
save_jsonl(train_phon_jsonl_data, os.path.join(ouput_base_dir, 'train', 'phoneme.jsonl'))
save_jsonl(dev_audio_jsonl_data, os.path.join(ouput_base_dir, 'val', 'audio.jsonl'))
save_jsonl(dev_phon_jsonl_data, os.path.join(ouput_base_dir, 'val', 'phoneme.jsonl'))
save_jsonl(test_audio_jsonl_data, os.path.join(ouput_base_dir, 'test', 'audio.jsonl'))
save_jsonl(test_phone_jsonl_data, os.path.join(ouput_base_dir, 'test', 'phoneme.jsonl'))

# 并行保存 H5 数据
save_h5_data('train', split2h5data['train'])
save_h5_data('val', split2h5data['val'])
save_h5_data('test', split2h5data['test'])
# with concurrent.futures.ThreadPoolExecutor() as executor:
#     executor.submit(save_h5_data, 'train', split2h5data['train'])
#     executor.submit(save_h5_data, 'val', split2h5data['val'])
#     executor.submit(save_h5_data, 'test', split2h5data['test'])


Found 301179 TextGrid files


In [ ]:
# 生成xvector.h5文件    
import wespeaker
import tqdm
from h5py import File
import os
dir_path='/nas-wulanchabu/jiahao.mei/data/tts/LibriTTS'
# dir_path='/nas-wulanchabu/jiahao.mei/data/tts/LibriTTS/test-other/367/130732'
all_files = []  # 这里应该存放 (dir, file) 的元组
for dir, _, file_names in os.walk(dir_path):
    for file in file_names:
        if file.endswith('.wav'):
            all_files.append((dir,file))  # 直接存储参数元组
print(f'Found {len(all_files)} files')
model=None
def extract_xvector(audio_path,lang='english'):
    global model
    if model is None:
        model = wespeaker.load_model(lang)
        model.set_device('cuda:0')
    xvector = model.extract_embedding(audio_path)
    return xvector
embeds={}

for (dir,file) in tqdm.tqdm(all_files):
    audio_path=os.path.join(dir,file)
    embed=extract_xvector(audio_path)
    embeds[file.replace('.wav','')]=embed
h5_save_path='/nas-wulanchabu/jiahao.mei/code/x2audio/data/libritts/xvector.h5'
with File(h5_save_path,'w') as hf:
    hf.create_group('xvector')
    for id in embeds.keys():
        hf['xvector'][id]=embeds[id]
print(f'save xvector to {h5_save_path}')

In [ ]:
# 修改已生成的 h5 文件，添加xvector字段
from h5py import File
import numpy as np
split_lis=['train','val','test']
for split in split_lis:
    content_or_path=f'/nas-wulanchabu/jiahao.mei/code/x2audio/data/libritts/bak/{split}/phone.h5'
    save_path=f'/nas-wulanchabu/jiahao.mei/code/x2audio/data/libritts/{split}/phone_vec.h5'
    xvector_path='/nas-wulanchabu/jiahao.mei/code/x2audio/data/libritts/xvector.h5'
    xvector={}
    with File(xvector_path,'r') as hf:
        xvector=hf['xvector']
        with File(content_or_path, "r") as hf:
            phoneme = hf["phoneme"]
            phoneme_duration = hf["phoneme_duration"]
            spkid = hf["spkid"]
            # new_spkid = {}
            # new_phoneme = {}
            # new_phoneme_duration = {}
            
            # #全部转换成字典
            # for k in  hf["spkid"].keys() :
            #     new_spkid[k] = '8051'
            #     new_phoneme[k] = phoneme[k][()]
            #     new_phoneme_duration[k] = phoneme_duration[k][()]
                
            # print(new_phoneme)
            with File(save_path, "w") as hf:
                hf.create_group("spkid")
                hf.create_group("phoneme")
                hf.create_group("phoneme_duration")
                hf.create_group("xvector")  
                for id in spkid.keys():
                    hf['xvector'][id]=xvector[id][()]
                    hf['spkid'][id] = spkid[id][()]
                    hf['phoneme'][id] = phoneme[id][()]
                    hf['phoneme_duration'][id] = phoneme_duration[id][()]
    print(f'save to {save_path}')
        

save to /nas-wulanchabu/jiahao.mei/code/x2audio/data/libritts/test/phone_vec.h5


In [7]:
#从train_clean_100 中生成50H训练数据
import os
import json
from h5py import File
import numpy as np

def get_wav_filenames(directory):
    """ 遍历文件夹，获取所有 .wav 文件的文件名，处理后返回列表 """
    wav_files = []
    
    for root, _, files in os.walk(directory):
        for file in files:
            if file.endswith(".wav"):
                filename = file[:-4]  # 去除.wav后缀
                parts = filename.split("_", 1)  # 分割最前面的数字部分
                
                if parts[0].isdigit() and len(parts[0]) < 4:
                    parts[0] = parts[0].zfill(4)  # 数字部分补全 4 位
                
                new_filename = "_".join(parts)
                wav_files.append(new_filename)
    
    return set(wav_files)  # 使用 set 方便快速查找

def filter_audio_jsonl(input_jsonl, output_jsonl, valid_ids):
    """ 读取 audio.jsonl 文件，仅保留 audio_id 在 valid_ids 中的记录，并保存 """
    with open(input_jsonl, "r", encoding="utf-8") as f, open(output_jsonl, "w", encoding="utf-8") as out_f:
        for line in f:
            data = json.loads(line)
            if data["audio_id"] in valid_ids:
                out_f.write(json.dumps(data) + "\n")
    
    print(f"Filtered JSONL saved to {output_jsonl}")

def filter_phoneme_h5(input_h5, output_h5, valid_ids):
    """ 读取 phoneme.h5 文件，仅保留 id 在 valid_ids 中的数据，并保存 """
    with File(input_h5, "r") as hf_in, File(output_h5, "w") as hf_out:
        hf_out.create_group("spkid")
        hf_out.create_group("phoneme")
        hf_out.create_group("phoneme_duration")
        hf_out.create_group("xvector")
        
        for dataset in ["spkid", "phoneme", "phoneme_duration", "xvector"]:
            if dataset in hf_in:
                for key in hf_in[dataset].keys():
                    if key in valid_ids:
                        hf_out[dataset][key] = hf_in[dataset][key][()]
    
    print(f"Filtered HDF5 saved to {output_h5}")

# 配置路径
wav_dir = "/cpfs_shared/jiahao.mei/data/tts/LibriTTS/train-clean-100"  # 修改为实际的 wav 文件目录
audio_jsonl = "/cpfs_shared/jiahao.mei/code/x2audio/data/libritts/train/audio.jsonl"  # 修改为 audio.jsonl 的路径
phoneme_h5 = "/cpfs_shared/jiahao.mei/code/x2audio/data/libritts/train/phone.h5"  # 修改为 phoneme.h5 的路径
phoneme_jsonl='/cpfs_shared/jiahao.mei/code/x2audio/data/libritts/train/phoneme.jsonl'

output__jsonl = "/cpfs_shared/jiahao.mei/code/x2audio/data/libritts/train_50h/audio.jsonl"  # 结果 JSONL 保存路径
output_h5 = "/cpfs_shared/jiahao.mei/code/x2audio/data/libritts/train_50h/"  # 结果 HDF5 保存路径

# 获取处理后的 WAV 文件名
valid_audio_ids = get_wav_filenames(wav_dir)

# 处理 JSONL 文件
filter_audio_jsonl(audio_jsonl, output_jsonl, valid_audio_ids)

# 处理 HDF5 文件
filter_phoneme_h5(phoneme_h5, output_h5, valid_audio_ids)

<Closed HDF5 group>

In [3]:

#从train_clean_100 中生成50H训练数据
import os
import json
from h5py import File
import numpy as np

def process_wav_filenames(folder_path):
    """
    遍历指定文件夹，获取所有 .wav 文件名并处理：
    1. 去除 .wav 后缀
    2. 文件名前面的数字部分补足 4 位
    """
    wav_list = []
    
    for root, _, files in os.walk(folder_path):
        for file in files:
            if file.endswith(".wav"):
                name, _ = os.path.splitext(file)
                parts = name.split("_")
                parts[0] = parts[0].zfill(4)  # 补足4位
                new_name = "_".join(parts)
                wav_list.append(new_name)
    print(f'len of wav_list:{len(wav_list)}')
    
    return set(wav_list)  # 返回一个集合，方便快速查找

def filter_jsonl(input_jsonl, output_jsonl, valid_ids):
    """
    读取 JSONL 文件（audio.jsonl 或 phoneme.jsonl），
    仅保留 audio_id 存在于 valid_ids 的数据，并保存到新的 JSONL 文件。
    """
    with open(input_jsonl, "r", encoding="utf-8") as infile, \
         open(output_jsonl, "w", encoding="utf-8") as outfile:
        for line in infile:
            data = json.loads(line)
            if data["audio_id"] in valid_ids:
                json.dump(data, outfile, ensure_ascii=False)
                outfile.write("\n")

def filter_h5(input_h5, output_h5, valid_ids):
    """
    读取 phoneme.h5 文件，筛选符合条件的 id，并保存为新的 H5 文件。
    """
    with File(input_h5, "r") as hf:
        phoneme = hf["phoneme"]
        phoneme_duration = hf["phoneme_duration"]
        spkid = hf["spkid"]
        xvector=hf["xvector"]

        with File(output_h5, "w") as new_hf:
            new_hf.create_group("spkid")
            new_hf.create_group("phoneme")
            new_hf.create_group("phoneme_duration")
            new_hf.create_group("xvector")

            for audio_id in spkid.keys():
                if audio_id in valid_ids:
                    new_hf["spkid"][audio_id] = spkid[audio_id][()]
                    new_hf["phoneme"][audio_id] = phoneme[audio_id][()]
                    new_hf["phoneme_duration"][audio_id] = phoneme_duration[audio_id][()]
                    bewew_xvector=xvector[audio_id][()] 

    print(f"Filtered H5 file saved to {output_h5}")

def main(wav_folder, audio_jsonl, phoneme_jsonl, phoneme_h5, output_audio_jsonl, output_phoneme_jsonl, output_h5):
    """
    主函数：执行所有处理步骤。
    """
    # 1. 处理 WAV 文件名
    valid_audio_ids = process_wav_filenames(wav_folder)
    
    # 2. 处理 audio.jsonl
    filter_jsonl(audio_jsonl, output_audio_jsonl, valid_audio_ids)
    
    # 3. 处理 phoneme.jsonl
    filter_jsonl(phoneme_jsonl, output_phoneme_jsonl, valid_audio_ids)
    
    # 4. 处理 phoneme.h5
    filter_h5(phoneme_h5, output_h5, valid_audio_ids)
    
    print("Processing completed successfully!")

# 示例调用

wav_folder = "/cpfs_shared/jiahao.mei/data/tts/LibriTTS/train-clean-100"
audio_jsonl = "/cpfs_shared/jiahao.mei/code/x2audio/data/libritts/train/audio.jsonl"  # 修改为 audio.jsonl 的路径
phoneme_h5 = "/cpfs_shared/jiahao.mei/code/x2audio/data/libritts/train/phone.h5"  # 修改为 phoneme.h5 的路径
phoneme_jsonl='/cpfs_shared/jiahao.mei/code/x2audio/data/libritts/train/phoneme.jsonl'

output_audio_jsonl = "/cpfs_shared/jiahao.mei/code/x2audio/data/libritts/train_50h/audio.jsonl"
output_phoneme_jsonl = "/cpfs_shared/jiahao.mei/code/x2audio/data/libritts/train_50h/phoneme.jsonl"
output_h5 = "/cpfs_shared/jiahao.mei/code/x2audio/data/libritts/train_50h/phone.h5"

main(wav_folder, audio_jsonl, phoneme_jsonl, phoneme_h5, output_audio_jsonl, output_phoneme_jsonl, output_h5)

len of wav_list:33236
Filtered H5 file saved to /cpfs_shared/jiahao.mei/code/x2audio/data/libritts/train_50h/phone.h5
Processing completed successfully!
